In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

c:\dev\predictive-maintenance-mlops


missing a column goes unnoticed

In [2]:
import io
import pandas as pd
import numpy as np
from turbofan.data import RAW_DIR, COLUMNS, load_cmapss

In [3]:
lines = (RAW_DIR / "train_FD001.txt").read_text().splitlines()[:3]
shortened = []

for line in lines:
    values = line.split()
    values = values[:-1]
    shortened.append(" ".join(values))

short = "\n".join(shortened)

df = load_cmapss(io.StringIO(short))
print(df.shape)
df[["sensor_20", "sensor_21"]]

(3, 26)


,sensor_20,sensor_21
0,39.06,NaN
1,39.00,NaN
2,38.95,NaN


missing values train without complaint

In [4]:
from sklearn.ensemble import RandomForestRegressor
from turbofan.data import add_rul
from turbofan.features import add_rolling_features, cap_rul, get_feature_columns

In [5]:
train = cap_rul(add_rul(load_cmapss(RAW_DIR / "train_FD001.txt")))

holey = train.copy()
rows_to_break = holey.sample(frac=0.05, random_state=0).index
holey.loc[rows_to_break, "sensor_11"] = np.nan
holey = add_rolling_features(holey)

print("missing values in features:", holey[get_feature_columns()].isna().sum().sum())

missing values in features: 1035


In [6]:
model = RandomForestRegressor(n_estimators=20, max_depth=8, random_state=0, n_jobs=-1)
model.fit(holey[get_feature_columns()], holey["rul"])
print("trained without complaint")

trained without complaint


In [7]:
def check_no_missing(df):
    missing = df.isna().sum().sum()
    if missing > 0:
        raise ValueError(f"data has {missing} missing values")
    return True

In [8]:
check_no_missing(train)
# check_no_missing(holey)
# check_no_missing(df)

True

a failure your check can’t catch

Missing values are one kind of bad data. Here’s another. sensor_2 is a temperature in degrees Rankine. Imagine the supplier switches it to Celsius without telling anyone.

In [9]:
print("normal range:", train["sensor_2"].min(), "to", train["sensor_2"].max())

celsius = train.copy()
celsius["sensor_2"] = (celsius["sensor_2"] - 491.67) * 5 / 9

print("after the change:", celsius["sensor_2"].min().round(1), "to", celsius["sensor_2"].max().round(1))

normal range: 641.21 to 644.53
after the change: 83.1 to 84.9


In [10]:
check_no_missing(celsius)

True

In [11]:
def check_range(df, column, low, high, margin=0.10):
    pad = (high - low) * margin
    low = low - pad
    high = high + pad
    too_low = (df[column] < low).sum()
    too_high = (df[column] > high).sum()
    bad = too_low + too_high
    if bad > 0:
        raise ValueError(f"{column}: {bad} values outside {low} to {high}")
    return True

In [12]:
check_range(train, "sensor_2", 641.21, 644.53)
check_range(celsius, "sensor_2", 641.21, 644.53)

ValueError: sensor_2: 20631 values outside 640.878 to 644.862

In [ ]:
drifted = train.copy()
drifted["sensor_2"] = drifted["sensor_2"] + 30
check_range(drifted, "sensor_2", 641.21, 644.53)

ValueError: sensor_2: 20631 values outside 640.878 to 644.862

From one sensor to all 21

In [ ]:
sensor_columns = [c for c in COLUMNS if c.startswith("sensor_")]
print(len(sensor_columns), sensor_columns[:3])

21 ['sensor_1', 'sensor_2', 'sensor_3']


In [ ]:
def build_profile(df, columns):
    profile = {}
    for column in columns:
        profile[column] = (df[column].min(), df[column].max())
    return profile

In [ ]:
profile = build_profile(train, sensor_columns)
print(profile["sensor_2"])
print(profile["sensor_1"])
print(profile["sensor_11"])

(np.float64(641.21), np.float64(644.53))
(np.float64(518.67), np.float64(518.67))
(np.float64(46.85), np.float64(48.53))


In [ ]:
def check_all_ranges(df, profile):
    problems = []
    for column, (low, high) in profile.items():
        try:
            check_range(df, column, low, high)
        except ValueError as error:
            problems.append(str(error))
    if problems:
        raise ValueError("range check failed:\n" + "\n".join(problems))
    return True

In [ ]:
two_faults = celsius.copy()
two_faults["sensor_11"] = two_faults["sensor_11"] + 5

check_all_ranges(two_faults, profile)

ValueError: range check failed:
sensor_2: 20631 values outside 640.878 to 644.862
sensor_11: 20631 values outside 46.682 to 48.698

Final check after building pipeline, turbofan/validate.py

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

from turbofan.data import RAW_DIR, COLUMNS, load_cmapss
from turbofan.validate import check_no_missing, build_profile, check_all_ranges

train = load_cmapss(RAW_DIR / "train_FD001.txt")
sensor_columns = [c for c in COLUMNS if c.startswith("sensor_")]
profile = build_profile(train, sensor_columns)

print(check_no_missing(train))
print(check_all_ranges(train, profile))

True
True
True


In [ ]:
broken = train.copy()
broken["sensor_2"] = (broken["sensor_2"] - 491.67) * 5 / 9
check_all_ranges(broken, profile)

ValueError: range check failed:
sensor_2: 20631 values outside 640.878 to 644.862

In [3]:
from turbofan.data import RAW_DIR, load_cmapss
from turbofan.validate import SENSOR_COLUMNS, build_profile, save_profile, load_profile, check_all_ranges, PROFILE_PATH

train = load_cmapss(RAW_DIR / "train_FD001.txt")
save_profile(build_profile(train, SENSOR_COLUMNS))

print(PROFILE_PATH.read_text()[:150])
print(check_all_ranges(train, load_profile()))

{
  "sensor_1": [
    518.67,
    518.67
  ],
  "sensor_2": [
    641.21,
    644.53
  ],
  "sensor_3": [
    1571.04,
    1616.91
  ],
  "sensor_4": 
True


In [4]:
import turbofan.validate

In [32]:
import numpy as np

def check_prediction_range(predictions, low, high):
    low_count = 0
    high_count = 0
    nan_count = np.isnan(predictions).sum()
  
    for pred in predictions:
        if pred < low:
            low_count += 1

        if pred > high:
            high_count += 1
    if low_count > 0 or high_count > 0 or nan_count > 0:
        raise ValueError(f"{low_count} values are below {low} range and {high_count} values are above {high} range. {nan_count} values are Nan.")
    return True

In [34]:
check_prediction_range(np.array([10, 50, 125]), 0, 125)
check_prediction_range(np.array([-3.9, 50, 120]), 0, 125)
check_prediction_range(np.array([10, 50, 179.6]), 0, 125)
check_prediction_range(np.array([10, np.nan, 50]), 0, 125)

ValueError: 1 values are below 0 range and 0 values are above 125 range. 0 values are Nan.

In [3]:
from turbofan.quality import check_prediction_range
import numpy as np
check_prediction_range(np.array([10, 50, 125]), 0, 125)

True